# Spam X: Level 2, useful examples

In notebook 08 a counter example was *any* message the filter lets through. Most of them are **useless to a scammer**: appended junk
(`gt lt ok my when`) is visible, and a disguised phone number does not connect. Level 2 asks for a **useful example**: a counter example
that still does the scammer's job and looks like a normal message. It is a **constrained optimisation**:

$$
\min_{x'}\ \text{cost}(x,x')\quad\text{s.t.}\quad f(x')=\text{ham},\ \ \text{payload}(x')=\text{payload}(x),\ \ \text{sim}\big(\text{see}(x),\text{see}(x')\big)\ge\tau,\ \ \text{visible edits}\le B
$$

* **payload**: phone numbers and short codes, links, amounts (`£900`, `150p`) and the *call to action* (`call`, `text`, `claim`...) stay intact;
* **meaning**: an *independent* embedder (never the filter under attack) says the visible text is still close to the original;
* **looks normal**: only two kinds of edit are allowed: **invisible** ones (Cyrillic look-alike letters, zero-width spaces) and **fluent** ones
  (a whole benign message, near-synonyms, a paraphrase by a language model). Random junk words are not.
* **visible edits** $\le B$: what a human would notice (invisible edits cost nothing).

**Why care.** The examples that survive these constraints are the ones a *campaign* can use: they work on a whole scam family,
they may transfer, and they need few queries. Measuring them tells the defender which family and which model to harden first.

> **Scope.** As before: our own toy filters, the public 2005 SMS corpus, no real numbers or links, no deployment. Notebook 10 turns
> the results into defences.

**Requirements.** The MiniLM server (the *judge of meaning*) must be running (`make llama-up`); the Nomic and MiniLM victims need their servers too
(`AAS_SLM=skip` leaves the two embedding victims out). Optional: the chat server (`AAS_CHAT=live`).

In [ ]:
import os
from collections import Counter

import numpy as np
from matplotlib import pyplot as plt

import attacklib as al
import spamlib as sl

plt.rcParams["figure.dpi"] = 100
texts, y = sl.load()
X_train, X_test, y_train, y_test = sl.split(texts, y)
spam_test = [t for t, lab in zip(X_test, y_test) if lab == 1]
family = dict(zip(texts, sl.spam_families(texts, y)))  # scam family of every spam message

WHICH = ["nb_counts", "tfidf_lr", "fasttext", "vectors_lr"]
if os.environ.get("AAS_SLM") != "skip":
    WHICH += ["minilm_lr", "nomic_lr"]
try:
    V = al.build_victims(X_train, y_train, which=WHICH)
except RuntimeError as exc:  # a llama.cpp server is not running: leave the embedding victims out
    print("embedding victims skipped:", exc)
    V = al.build_victims(X_train, y_train, which=[k for k in WHICH if k not in ("minilm_lr", "nomic_lr")])
CHEAP = [k for k in V if k not in ("minilm_lr", "nomic_lr")]
N_MESSAGES = 40  # attacked messages: 40 keep the run to about 15 minutes; raise it for tighter numbers
messages = [t for t in spam_test if al.payload_spans(t)][:N_MESSAGES]
print(f"{len(messages)} test spam messages with a payload are attacked; victims: {[v.name for v in V.values()]}")

## A. The payload guard

`payload` finds the items that must survive a rewrite; `call_to_action` the verbs; `payload_intact` compares a rewrite with the original.

In [ ]:
for m in messages[:3]:
    print(repr(m[:110]))
    print("   payload:", al.payload(m), "| call to action:", sorted(al.call_to_action(m)))
demo = al.disguise_text(messages[0], al.payload(messages[0]), "leet", protect=False)
print("\ndisguising the payload itself:", repr(demo[:110]))
print("   payload intact?", al.payload_intact(messages[0], demo))

Disguising the words of the payload breaks it: a phone number written with letters does not dial. How many of the **level 1** disguises of
notebook 08 (top-25 words, no guard) break the payload or the call to action?

In [ ]:
lr_pipe = V["tfidf_lr"].encoder
feat = np.array(lr_pipe[0].get_feature_names_out())
TOP = [w for w in feat[np.argsort(-lr_pipe[1].coef_[0])] if w.isalpha()][:60]
print(f"{'level 1 disguise':18}{'payload or CTA broken':>24}")
for mode in ("leet", "dots", "space", "vowel", "homoglyph", "zwsp"):
    broken = np.mean([not al.payload_intact(m, al.disguise_text(m, TOP[:25], mode, protect=False)) for m in messages])
    print(f"{mode:18}{broken:24.0%}")

Leet, dots and spaces break the call to action (`c4ll`, `c.a.l.l`) in many messages; the two **invisible** disguises (homoglyph, zero-width) mostly
break it too when applied blindly, because they hit `call`, `text` and the digits. With `protect=True` the payload and the call to
action are simply skipped by the operators. What does a human see?

In [ ]:
w = "claim your free prize"
for mode in ("homoglyph", "zwsp"):
    d = al.disguise_text(w, ["claim", "free", "prize"], mode)
    print(f"{mode:10} raw {d!r:32} visible {al.visible(d)!r}   same as the original: {al.visible(d) == w}")

## B. The operators of a useful attack

1. **invisible**: Cyrillic look-alikes or zero-width spaces inside the spammy words (a human reads the same text);
2. **synonym**: replace one word by a near-synonym from a word-vector space (class 3 material: **pre-trained Word2Vec** as the attacker's
   dictionary; respellings such as `cost` -> `coast` are filtered out);
3. **camouflage**: prepend or append a whole benign message (a real ham message: fluent by construction).

The **judge** of meaning is MiniLM (Nomic when the victim is MiniLM), the **budget** 15 visible words, the threshold $\tau=0.7$.

In [ ]:
SENTENCES = al.benign_sentences(X_train, y_train, n=20)
print("camouflage examples:", SENTENCES[:3])

if os.environ.get("AAS_W2V") == "skip":  # no 1.7 GB download: in-domain skip-gram vectors instead
    from gensim.models import Word2Vec

    kv = Word2Vec(
        [sl.tokenize(t) for t in X_train], vector_size=100, window=5, min_count=2, sg=1, epochs=30, workers=1, seed=42
    ).wv
else:
    kv = sl.load_pretrained_word2vec()
print("neighbours of `prize`:", [w for w, _ in kv.most_similar("prize", topn=6)])

OPS = {
    "invisible": al.op_disguise(TOP, modes=("homoglyph", "zwsp"), whole=True),
    "synonym": al.op_substitute(kv),
    "camouflage": al.op_camouflage(SENTENCES),
}
TAU, MAX_EDITS, STEPS = 0.7, 15, 6


def attack(key, msgs, ops, tau=TAU, max_edits=MAX_EDITS, steps=STEPS):
    v = V[key]
    cons = al.Constraints(payload=True, judge=al.judge_for(v), tau=tau, max_edits=max_edits)
    return [al.greedy(v, m, [OPS[o] if isinstance(o, str) else o for o in ops], cons, steps) for m in msgs], cons

## C. Level 1 versus level 2: the price of being useful

Same messages, same victims. **Level 1**: append up to 12 common ham words or disguise words without any guard (notebook 08). **Level 2**:
the three operators above under the constraints. The table shows the ASR of level 1, the share of those examples that still have their
payload, and the ASR of level 2 (every level 2 success satisfies all the constraints by construction).

In [ ]:
ham_words = [w for t, lab in zip(X_train, y_train) if lab == 0 for w in sl.tokenize(t) if w.isalpha() and len(w) > 1]
uniq, counts = np.unique(ham_words, return_counts=True)
CANDIDATES = [str(w) for w in uniq[np.argsort(-counts)][:40]]
level1_ops = [
    al.op_append(CANDIDATES),
    al.op_disguise(TOP[:25], modes=("leet", "dots", "homoglyph"), protect=False, whole=True),
]

N_SLM = 10
rows, results = {}, {}
for key, v in V.items():
    print("attacking", v.name, flush=True)
    msgs = messages if key in CHEAP else messages[:N_SLM]
    l1 = [al.greedy(v, m, level1_ops, al.Constraints(max_edits=99), max_steps=12) for m in msgs]
    l1_win = [r for r in l1 if r.success and r.score_before >= 0]
    caught = [r for r in l1 if r.score_before >= 0]
    l2, cons = attack(key, msgs, ["invisible", "synonym", "camouflage"])
    results[key] = (msgs, l2)
    s2 = al.summarise(l2, cons)
    keep = [r for r in l1_win if al.payload_intact(r.original, r.adversarial)]
    rows[key] = (
        len(l1_win) / len(caught),
        len(keep) / len(caught),
        s2["useful_asr"],
        s2["edits"],
        s2["queries"],
    )
print(f"{'victim':26}{'L1 ASR':>8}{'L1 + payload':>14}{'L2 useful ASR':>15}{'visible edits':>15}{'queries':>9}")
for key, (a, b, c, e, q) in rows.items():
    print(f"{V[key].name:26}{a:8.2f}{b:14.2f}{c:15.2f}{e:15.1f}{q:9.0f}")

Read it in three steps. (1) The **headline ASR of level 1** (0.60 to 1.00) **overstates the threat**: once we ask that the payload survives, only
0.11 to 0.38 of the caught spam has a working level 1 example (*L1 + payload*): the blind disguises break the phone number or the call to action,
and the rest carries visible junk. (2) **Level 2** is built for usefulness and evades 0.51 (Naive Bayes) to 1.00 (MiniLM) of the messages *with the
payload, the meaning and at most 15 visible words intact*. (3) The embedders are the **hardest for junk words** (0.60 to 0.67) but the **easiest for fluent
camouflage**: two or three well-chosen benign sentences move a mean-pooled embedding. Usefulness does not simply lower the ASR; it changes **which
attacks work**.

## D. Which operator does the work?

The same attack, one operator at a time (and all together), useful ASR:

In [ ]:
sets = {
    "invisible": ["invisible"],
    "synonym": ["synonym"],
    "camouflage": ["camouflage"],
    "all three": ["invisible", "synonym", "camouflage"],
}
table = {}
for key in V:
    msgs = messages if key in CHEAP else messages[:N_SLM]
    table[key] = {}
    for name, ops in sets.items():
        res, cons = results[key][1], None
        if name != "all three":
            res, cons = attack(key, msgs, ops)
            table[key][name] = al.summarise(res, cons)["useful_asr"]
        else:
            table[key][name] = al.summarise(
                results[key][1], al.Constraints(payload=True, judge=al.judge_for(V[key]), tau=TAU, max_edits=MAX_EDITS)
            )["useful_asr"]
print(f"{'victim':26}" + "".join(f"{n:>12}" for n in sets))
for key, row in table.items():
    print(f"{V[key].name:26}" + "".join(f"{row[n]:12.2f}" for n in sets))

fig, ax = plt.subplots(figsize=(8, 3.8))
width = 0.2
for i, name in enumerate(sets):
    ax.bar(np.arange(len(V)) + i * width, [table[k][name] for k in V], width, label=name)
ax.set_xticks(
    np.arange(len(V)) + 1.5 * width, [V[k].name.replace(" + ", "\n+ ").replace(" (", "\n(") for k in V], fontsize=7
)
ax.set(ylabel="useful ASR", ylim=(0, 1))
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Camouflage** is the strongest single operator (it is the *dilution* of notebook 08 made fluent), **synonyms** matter for the embedders (they react to
*meaning*: 0.40 to 0.56), and **invisible** edits alone barely move any score (0.00 to 0.10: they hide only a few words). The operators **add up**:
for Naive Bayes 0.05, 0.08 and 0.18 alone become 0.51 together.

## E. A paraphrase by a small language model

The last operator asks a **chat model** for a rewrite. The payload is replaced by placeholders (`[P1]`), the model paraphrases, the placeholders are
restored: the payload survives by construction. The table shipped with the course (`datasets/spam_paraphrases.json`) holds the paraphrases of
these test messages made with **Qwen3.5-9B** (llama.cpp, Vulkan; the model that won the benchmark of the class-04 preparation, see
`llama/README.md`). Set `AAS_CHAT=live` with the `chat` service running to generate your own (about 7 s per message).

In [ ]:
table_para = al.load_paraphrases()
if os.environ.get("AAS_CHAT") == "live":
    table_para = al.paraphrase(messages[:20], n=2, style="casual")
have = [m for m in messages if table_para.get(m)]
print(f"{len(have)} of {len(messages)} messages have a stored paraphrase")
for m in have[:2]:
    print("-", m[:110], "\n  ->", table_para[m][0][:130])

In [ ]:
para_rows = {}
for key, v in V.items():
    msgs = [m for m in have if key in CHEAP or m in messages[:N_SLM]] or have[:N_SLM]
    cons = al.Constraints(payload=True, judge=al.judge_for(v), tau=TAU, max_edits=40)
    alone = [al.greedy(v, m, [al.op_paraphrase(table_para)], cons, 1) for m in msgs]
    combo = [
        al.greedy(v, m, [al.op_paraphrase(table_para), OPS["camouflage"], OPS["synonym"], OPS["invisible"]], cons, 4)
        for m in msgs
    ]
    para_rows[key] = (al.summarise(alone, cons), al.summarise(combo, cons))
print(f"{'victim':26}{'paraphrase alone':>18}{'+ camouflage, synonyms, invisible':>36}")
for key, (a, b) in para_rows.items():
    print(f"{V[key].name:26}{a.get('useful_asr', float('nan')):18.2f}{b.get('useful_asr', float('nan')):36.2f}")

A paraphrase **alone** rarely evades (0.00 to 0.21): the rewrite keeps the spam vocabulary (`prize`, `claim`, `call`). Its value is as one more
*operator*: with a 40-word budget the search combines it with camouflage, synonyms and invisible edits and reaches 0.89 to 1.00. (The model declined or
broke a placeholder in a few percent of the cases; those messages have no paraphrase and the search falls back on the other operators.)

## F. What a campaign gains: scam families

A campaign sends the *same* scam to many people, so what matters is not one message but a **family**. Useful ASR of the combined attack
by scam family (the word-based victims, where the sample is large enough), and the type of edit that won:

In [ ]:
def kind(orig, adv):
    used = []
    if al.visible(adv) != adv:
        used.append("invisible")
    base = al.visible(adv)
    for s in SENTENCES:
        if s in base:
            used.append("camouflage")
            base = base.replace(s, "").strip()
    if base.split() != orig.split():
        used.append("synonym")
    return used or ["-"]


print(f"{'family':20}{'n':>4}" + "".join(f"{V[k].name[:14]:>16}" for k in CHEAP) + "   winning edits (tf-idf + LR)")
for f, name in enumerate(sl.FAMILY_NAMES):
    line = f"{name:20}"
    n_f = sum(family[m] == f for m in messages)
    line += f"{n_f:4d}"
    for key in CHEAP:
        msgs, res = results[key]
        sel = [r for r in res if family[r.original] == f and r.score_before >= 0]
        line += f"{np.mean([r.success for r in sel]) if sel else float('nan'):16.2f}"
    msgs, res = results["tfidf_lr"]
    kinds = Counter(k for r in res if family[r.original] == f and r.success for k in kind(r.original, r.adversarial))
    print(line + "   " + ", ".join(f"{k} {c}" for k, c in kinds.most_common(3)))

Some families fall almost always (account statements, mobile upgrades: 0.6 to 1.0 for every filter), others resist (ringtone subscriptions: 0.0 to 0.67,
on only 3 messages). With 3 to 12 messages per family these are hints, not estimates: exercise 4 asks for more. The **recipe** differs per family (camouflage
wins in every family, invisible edits help next), which is what a campaign operator would tune and what a defender should look at first.

**Do the useful examples transfer?** Take the successful examples crafted against one victim and test them on the others:

In [ ]:
adv = {k: [r.adversarial for r in results[k][1] if r.success and r.score_before >= 0] for k in V}
adv = {k: a for k, a in adv.items() if a}
tm = al.transfer_matrix(adv, V)
print(f"{'crafted against':26}" + "".join(f"{v.name[:14]:>16}" for v in V.values()))
for src, row in tm.items():
    print(f"{V[src].name:26}" + "".join(f"{row[dst]:16.2f}" for dst in V))

Camouflage and synonyms are **model-independent** edits, so useful examples transfer better than the level 1 ones of notebook 08: examples crafted against
TF-IDF + LR also beat fastText vectors 0.80 of the time, those crafted against Nomic beat MiniLM 0.78 and the word vectors 0.56. **The stronger the model
the attacker had to beat, the more transferable the example**, which is what makes the campaign cheap: craft once against a public surrogate, send to everybody.

## Take-aways

* A counter example is a **blind spot**; a **useful** example also keeps the payload, the meaning and a normal look. That does not just lower the ASR: it decides which attacks work.
* The cheap operators are the ones that dilute or hide evidence without touching the payload: benign camouflage, synonyms, invisible characters.
* A paraphrase model is one more operator, not a magic wand: alone it barely evades, in the search it helps.
* Look at **families**: campaigns reuse a recipe, so defences and evaluations must be per family, not per message.

## Exercises

1. Raise the threshold $\tau$ to 0.85 and lower it to 0.5. How do the useful ASR and the shape of the examples change? What does $\tau$ mean for a scammer?
2. Replace the judge by Nomic for all victims. Does the result depend on which embedder judges the meaning?
3. Give the camouflage operator 200 sentences instead of 20, or choose the sentences *nearest* to the spam message (MiniLM). Which is better, and why?
4. The per-family table has 3 to 12 messages per family: raise `N_MESSAGES` to 120 and see which families stay easy.
5. Add a defence *inside* the search: reject candidates that contain a zero-width character or a Cyrillic letter. Which operator loses?